In [1]:
print("In the name of Allah")

In the name of Allah


https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset

In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.
Path to dataset files: /kaggle/input/sms-spam-collection-dataset


In [10]:
import pandas as pd

In [11]:
df = pd.read_csv("/kaggle/input/sms-spam-collection-dataset/spam.csv", encoding="latin-1")

In [12]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [13]:
df = df[["v1", "v2"]]
df.columns = ["label", "text"]
print(df.shape)

(5572, 2)


In [14]:
df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [15]:
# Problem 1

import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [16]:
df["clean"] = df["text"].str.lower()
df["clean"].head()

,clean
0,"go until jurong point, crazy.. available only ..."
1,ok lar... joking wif u oni...
2,free entry in 2 a wkly comp to win fa cup fina...
3,u dun say so early hor... u c already then say...
4,"nah i don't think he goes to usf, he lives aro..."


In [17]:
def remove_html_tags(text):
    pattern = re.compile(r"<.*?>")
    return pattern.sub(r"", text)

df["clean"] = df["clean"].apply(remove_html_tags)

In [18]:
def remove_url(text):
    pattern = re.compile(r"https?://\S+|www\.\S+")
    return pattern.sub(r"", text)

df["clean"] = df["clean"].apply(remove_url)

In [19]:
def remove_emoji(text):
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U00002700-\U000027BF"  # dingbats
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(r"", text)

df["clean"] = df["clean"].apply(remove_emoji)

In [20]:
chat_words = {
    "AFAIK": "As Far As I Know",
    "ASAP": "As Soon As Possible",
    "ATM": "At The Moment",
    "BRB": "Be Right Back",
    "BTW": "By The Way",
    "CU": "See You",
    "CUL8R": "See You Later",
    "FYI": "For Your Information",
    "GR8": "Great",
    "IDK": "I Do Not Know",
    "IMO": "In My Opinion",
    "LOL": "Laughing Out Loud",
    "OMG": "Oh My God",
    "PLZ": "Please",
    "PLS": "Please",
    "THX": "Thanks",
    "TXT": "Text",
    "U": "You",
    "UR": "Your",
    "B4": "Before",
    "2DAY": "Today",
    "2MORO": "Tomorrow",
}

def chat_conversion(text):
    new_text = []
    for w in text.split():
        if w.upper() in chat_words:
            new_text.append(chat_words[w.upper()])
        else:
            new_text.append(w)
    return " ".join(new_text).lower()   # lowercase again because the replacements are capitalised

df["clean"] = df["clean"].apply(chat_conversion)

In [21]:
exclude = string.punctuation

def remove_punc(text):
    return text.translate(str.maketrans("", "", exclude))

df["clean"] = df["clean"].apply(remove_punc)

In [22]:
stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    cleaned = []
    for word in text.split():
        word = word.lower().strip(string.punctuation)
        if word and word not in stop_words:
            cleaned.append(word)
    return " ".join(cleaned)

df["clean"] = df["clean"].apply(remove_stopwords)

In [23]:
wordnet_lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    return " ".join([wordnet_lemmatizer.lemmatize(w, pos="v") for w in text.split()])

df["clean"] = df["clean"].apply(lemmatize_text)

In [24]:
pd.set_option("display.max_colwidth", 120)
df[["text", "clean"]].sample(8, random_state=42)

,text,clean
3245,"Funny fact Nobody teaches volcanoes 2 erupt, tsunamis 2 arise, hurricanes 2 sway aroundn no 1 teaches hw 2 choose a ...",funny fact nobody teach volcanoes 2 erupt tsunamis 2 arise hurricanes 2 sway aroundn 1 teach hw 2 choose wife natura...
944,I sent my scores to sophas and i had to do secondary application for a few schools. I think if you are thinking of a...,send score sophas secondary application school think think apply research cost also contact joke ogunrinde school on...
1044,"We know someone who you know that fancies you. Call 09058097218 to find out who. POBox 6, LS15HB 150p",know someone know fancy call 09058097218 find pobox 6 ls15hb 150p
2484,Only if you promise your getting out as SOON as you can. And you'll text me in the morning to let me know you made i...,promise get soon youll text morning let know make ok
812,Congratulations ur awarded either å£500 of CD gift vouchers & Free entry 2 our å£100 weekly draw txt MUSIC to 87066 ...,congratulations award either å£500 cd gift vouchers free entry 2 å£100 weekly draw text music 87066 tncs
2973,"I'll text carlos and let you know, hang on",ill text carlos let know hang
2991,K.i did't see you.:)k:)where are you now?,ki didt see youkwhere
2942,No message..no responce..what happend?,messageno responcewhat happend


In [25]:
empty = (df["clean"].str.strip() == "").sum()
print("Empty messages after cleaning:", empty)

# Drop them so the later problems (vectorizers) don't get empty documents
df = df[df["clean"].str.strip() != ""].reset_index(drop=True)
print("Final shape:", df.shape)

Empty messages after cleaning: 6
Final shape: (5566, 3)


In [26]:
# Problem 2

total_words = 0
vocab = set()

for sentence in df["clean"]:
    words = sentence.split()
    total_words += len(words)
    vocab.update(words)

print("Total number of words in the corpus :", total_words)
print("Number of unique words (vocabulary) :", len(vocab))

Total number of words in the corpus : 50910
Number of unique words (vocabulary) : 8413


In [27]:
# Problem 3

import numpy as np

vocab_list = sorted(vocab)
word2idx = {w: i for i, w in enumerate(vocab_list)}
V = len(vocab_list)

def one_hot_sentence(sentence):
    """Returns a matrix of shape (num_words_in_sentence, V) – one row per word."""
    words = sentence.split()
    mat = np.zeros((len(words), V), dtype=np.int8)
    for row, w in enumerate(words):
        mat[row, word2idx[w]] = 1
    return mat

sample_sentence = df["clean"][0]
print("Sentence:", sample_sentence)

oh = one_hot_sentence(sample_sentence)
print("One-hot matrix shape:", oh.shape)

# show where each word's "1" is
for w, row in zip(sample_sentence.split(), oh):
    print(f"{w:15} -> index {row.argmax()}")

Sentence: go jurong point crazy available bugis n great world la e buffet cine get amore wat
One-hot matrix shape: (16, 8413)
go              -> index 3407
jurong          -> index 4219
point           -> index 5783
crazy           -> index 2275
available       -> index 1335
bugis           -> index 1747
n               -> index 5087
great           -> index 3499
world           -> index 8143
la              -> index 4360
e               -> index 2748
buffet          -> index 1745
cine            -> index 2038
get             -> index 3366
amore           -> index 1123
wat             -> index 7931


In [28]:
from sklearn.feature_extraction.text import CountVectorizer

ohe = CountVectorizer(binary=True)
X_ohe = ohe.fit_transform(df["clean"])

print("Shape (documents x vocabulary):", X_ohe.shape)

pd.DataFrame(
    X_ohe[:5].toarray(),
    columns=ohe.get_feature_names_out()
).loc[:, lambda d: d.sum() > 0]   # show only the columns that are active in these 5 rows

Shape (documents x vocabulary): (5566, 8359)


,08452810075over18s,2005,21st,87121,already,amore,apply,around,available,buffet,...,text,think,though,tkts,usf,wat,wif,win,wkly,world
0,0,0,0,0,0,1,0,0,1,1,...,0,0,0,0,0,1,0,0,0,1
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2,1,1,1,1,0,0,1,0,0,0,...,1,0,0,1,0,0,0,1,1,0
3,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,1,0,0,...,0,1,1,0,1,0,0,0,0,0


In [29]:
# Problem 4

cv = CountVectorizer()
X_bow = cv.fit_transform(df["clean"])

print("BoW matrix shape:", X_bow.shape)
print("Vocabulary size :", len(cv.vocabulary_))

BoW matrix shape: (5566, 8359)
Vocabulary size : 8359


In [30]:
# Vocabulary (word -> column index) – first 15 entries
list(cv.vocabulary_.items())[:15]

[('go', 3423),
 ('jurong', 4227),
 ('point', 5788),
 ('crazy', 2294),
 ('available', 1361),
 ('bugis', 1771),
 ('great', 3515),
 ('world', 8144),
 ('la', 4366),
 ('buffet', 1769),
 ('cine', 2058),
 ('get', 3382),
 ('amore', 1149),
 ('wat', 7933),
 ('ok', 5394)]

In [31]:
# Times each word has occurred in the whole corpus = column sums of the BoW matrix
word_counts = np.asarray(X_bow.sum(axis=0)).ravel()

counts_df = pd.DataFrame({
    "word": cv.get_feature_names_out(),
    "count": word_counts
}).sort_values("count", ascending=False).reset_index(drop=True)

print("Top 20 most frequent words:")
counts_df.head(20)

Top 20 most frequent words:


,word,count
0,get,688
1,call,653
2,go,551
3,im,462
4,text,337
5,come,324
6,know,285
7,dont,279
8,ok,277
9,free,276


In [32]:
# Sanity check: sum of all counts vs the total words from Problem 2
print("Default CountVectorizer :", counts_df["count"].sum(), "| Problem 2 total:", total_words)

# Default token_pattern drops 1-character tokens and symbols like the pound sign.
# token_pattern=r"\S+" mirrors Python's .split(), so the totals should match exactly:
cv_all = CountVectorizer(token_pattern=r"\S+")
X_all = cv_all.fit_transform(df["clean"])
print("With split()-style tokens:", X_all.sum(), "| vocab:", len(cv_all.vocabulary_), "(Problem 2 vocab:", len(vocab), ")")

Default CountVectorizer : 48847 | Problem 2 total: 50910
With split()-style tokens: 50910 | vocab: 8413 (Problem 2 vocab: 8413 )


In [33]:
# Problem 5

configs = {
    "unigram (1,1)": (1, 1),
    "bigram  (2,2)": (2, 2),
    "trigram (3,3)": (3, 3),
}

rows = []
for name, ngram in configs.items():
    vec = CountVectorizer(ngram_range=ngram)
    X = vec.fit_transform(df["clean"])
    sparsity = 100 * (1 - X.nnz / (X.shape[0] * X.shape[1]))
    rows.append({
        "type": name,
        "vocab size (dimensions)": X.shape[1],
        "non-zero entries": X.nnz,
        "sparsity %": round(sparsity, 4),
    })
    if ngram == (2, 2):
        bigram_vec, X_bi = vec, X
    if ngram == (3, 3):
        trigram_vec, X_tri = vec, X

pd.DataFrame(rows)

,type,vocab size (dimensions),non-zero entries,sparsity %
0,"unigram (1,1)",8359,46633,99.8998
1,"bigram (2,2)",30778,43141,99.9748
2,"trigram (3,3)",31234,37831,99.9782


In [34]:
# A peek at the most frequent bi-grams and tri-grams
def top_ngrams(vec, X, n=10):
    counts = np.asarray(X.sum(axis=0)).ravel()
    return (pd.DataFrame({"ngram": vec.get_feature_names_out(), "count": counts})
              .sort_values("count", ascending=False).head(n).reset_index(drop=True))

print("Top bi-grams");  display(top_ngrams(bigram_vec, X_bi))
print("Top tri-grams"); display(top_ngrams(trigram_vec, X_tri))

Top bi-grams


,ngram,count
0,please call,64
1,laugh loud,59
2,call later,52
3,ill call,48
4,let know,42
5,sorry ill,39
6,dont know,38
7,good morning,30
8,customer service,29
9,try contact,28


Top tri-grams


,ngram,count
0,ill call later,42
1,sorry ill call,38
2,prize guarantee call,21
3,happy new year,18
4,call customer service,16
5,reply call 08000930705,14
6,land line claim,14
7,2003 account statement,13
8,please send message,13
9,private 2003 account,13


In [35]:
# Problem 6

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df["clean"])

print("TF-IDF matrix shape:", X_tfidf.shape)

TF-IDF matrix shape: (5566, 8359)


In [36]:
idf_df = pd.DataFrame({
    "word": tfidf.get_feature_names_out(),
    "idf": tfidf.idf_
})

print("Lowest IDF (most common words, least informative):")
display(idf_df.sort_values("idf").head(15).reset_index(drop=True))

print("Highest IDF (rarest words – appear in only 1 message):")
display(idf_df.sort_values("idf", ascending=False).head(15).reset_index(drop=True))

Lowest IDF (most common words, least informative):


,word,idf
0,get,3.170987
1,call,3.241105
2,go,3.410003
3,im,3.577239
4,come,3.910879
5,text,3.917501
6,ok,4.022493
7,know,4.041115
8,send,4.063930
9,dont,4.107159


Highest IDF (rarest words – appear in only 1 message):


,word,idf
0,ûówell,8.931464
1,ûïharry,8.931464
2,0089my,8.931464
3,0121,8.931464
4,01223585236,8.931464
5,zouk,8.931464
6,zoom,8.931464
7,zogtorius,8.931464
8,zindgi,8.931464
9,zhong,8.931464


In [37]:
# Same print style as the tfidf notebook (first 15 words)
for word, idf in list(zip(tfidf.get_feature_names_out(), tfidf.idf_))[:15]:
    print(f"{word:15} -> {idf:.4f}")

008704050406    -> 8.5260
0089my          -> 8.9315
0121            -> 8.9315
01223585236     -> 8.9315
01223585334     -> 8.5260
0125698789      -> 8.9315
02              -> 8.9315
020603          -> 8.0152
0207            -> 8.5260
02070836089     -> 8.9315
02072069400     -> 8.9315
02073162414     -> 8.5260
02085076972     -> 8.9315
020903          -> 8.5260
021             -> 8.5260
